In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chisquare
import json
from scipy.stats import chisquare, pearsonr, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [2]:
# Cargar los datos
df = pd.read_excel("../2. data/0 TimeLog (original, no modificar).xlsx")

In [3]:
# Copio para no modificar el original
tl = df.copy()
# Agrego LOS a cada fila
tl["LOS"] = (tl["TF"] - tl["TI"])

In [5]:
def empirical_counts(unidad, grd, hospital, tl=tl):
    # De aqui en adelante se reemplaza en la función
    tl_u = tl[tl["UNIDAD"].isin(["ICU", "SDU_WARD"])]
    v1 = tl_u[(tl_u["UNIDAD"] == unidad) & (tl_u["MS_GRD"] == grd) & (tl_u["HOSPITAL"] == f"Hospital_{hospital}")]
    vector = v1["LOS"].value_counts().reset_index().sort_values(by="LOS")
    vector.columns = ["LOS", "count"]
    vector["LOS"] = vector["LOS"] // 12
    # Reindexamos para que todo el soporte tenga al menos un valor llenando los ceros con 1
    values = np.arange(int(min(vector["LOS"])), int(max(vector["LOS"])) + 1) # Soporte para el KDE
    vector_filled = vector.set_index("LOS").reindex(values, fill_value=1).reset_index()
    empirical_counts = vector_filled["count"].values
    return empirical_counts

plot = False
resultados = {}
# Separando por hospital
for hospital in range(1, 4):
    resultados[hospital] = {}
    for unidad in ["ICU", "SDU_WARD"]:
        resultados[hospital][unidad] = {}
        for grd in range(1, 9):
            final_kde_pmf = empirical_counts(unidad, grd, hospital)/ empirical_counts(unidad, grd, hospital).sum()
            resultados[hospital][unidad][grd] = {
                "final_kde_pmf": final_kde_pmf.tolist()
            }
            print(f"Hospital {hospital} Unidad {unidad} GRD {grd}, {final_kde_pmf}")
            

Hospital 1 Unidad ICU GRD 1, [0.05977811 0.12750455 0.1318099  0.11756913 0.08991555 0.07285974
 0.05679748 0.06888558 0.03742341 0.02930949 0.02367942 0.02318265
 0.01573108 0.01854612 0.01854612 0.01142573 0.01092896 0.00910747
 0.00827952 0.00728597 0.00728597 0.00463653 0.00827952 0.00298063
 0.00364299 0.00314622 0.00248385 0.00264945 0.00248385 0.00480212
 0.00215267 0.00182149 0.00149031 0.00115913 0.00099354 0.00033118
 0.00082795 0.00082795 0.00082795 0.00149031 0.00115913 0.00049677
 0.00099354 0.00099354 0.00049677 0.00066236 0.00049677 0.00049677
 0.00016559 0.00066236 0.00049677]
Hospital 1 Unidad ICU GRD 2, [0.01892225 0.0678733  0.08658988 0.0855615  0.07239819 0.07219251
 0.06190868 0.08247635 0.04709996 0.0396956  0.03311394 0.02879473
 0.02344714 0.02406417 0.02879473 0.01830522 0.01542575 0.0123406
 0.01316331 0.01522007 0.01131222 0.01275195 0.01542575 0.00719868
 0.01110654 0.00987248 0.00863842 0.00534759 0.00575895 0.00843274
 0.00431921 0.00329083 0.00514192 0.0

In [6]:
def save_dict_as_json(data_dict, filename, folder):
    os.makedirs(folder, exist_ok=True)  # Create folder if it doesn't exist
    path = os.path.join(folder, filename)
    with open(path, 'w') as f:
        json.dump(data_dict, f, indent=4)
    print(f"Dictionary saved to: {path}")

In [7]:
# Save the results to a JSON file
save_dict_as_json(resultados, filename="los_empirico.json", folder="resultados incertidumbre")

Dictionary saved to: resultados incertidumbre/los_empirico.json
